# Scout ETL: Exploring Overture Maps Data

This notebook explores the raw Overture Maps `places` dataset for San Francisco **before** we run the production ETL pipeline in `02_build_sf_places.py`.

The goal is to:
- Understand the raw schema (what columns exist, what's nested)
- Sample real records to see what the data looks like
- Inventory the category values — these become the **closed vocabulary** in our LLM system prompt
- Assess data quality (completeness, confidence scores)

**Key idea**: We query Overture's S3 bucket directly using DuckDB's `httpfs` extension. No download required.

In [1]:
import duckdb
import pandas as pd

# --- CONFIGURATION ---
# Update RELEASE to the latest Overture Maps release if this URL stops working.
# See: https://docs.overturemaps.org/release/latest/
RELEASE = "2026-01-21.0"
PLACES_URL = f"s3://overturemaps-us-west-2/release/{RELEASE}/theme=places/type=place/*"

# San Francisco bounding box (WGS84)
SF_XMIN, SF_XMAX = -122.52, -122.35
SF_YMIN, SF_YMAX = 37.70, 37.84

# --- ENGINE SETUP ---
con = duckdb.connect()

# httpfs: lets DuckDB read Parquet files directly from S3 over HTTP
con.execute("INSTALL httpfs; LOAD httpfs;")
# spatial: adds ST_* geometry functions (GEOS-powered, same as PostGIS)
con.execute("INSTALL spatial; LOAD spatial;")

# Overture Maps is a public bucket — no AWS credentials required
con.execute("SET s3_region='us-west-2';")
con.execute("SET s3_access_key_id='';")
con.execute("SET s3_secret_access_key='';")

print(f"DuckDB version: {duckdb.__version__}")
print(f"Overture release: {RELEASE}")
print(f"SF bbox: ({SF_XMIN}, {SF_YMIN}) → ({SF_XMAX}, {SF_YMAX})")

DuckDB version: 1.4.4
Overture release: 2026-01-21.0
SF bbox: (-122.52, 37.7) → (-122.35, 37.84)


## 1. Schema Inspection

The Overture Places schema is rich — lots of nested structs. We need to understand it before we can decide what to keep. The `LIMIT 1` plus a tight bbox filter makes this nearly instant: DuckDB uses the Parquet row-group statistics to skip most of the global dataset before sending any data over the wire.

In [3]:
schema_df = con.execute(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{PLACES_URL}')
    WHERE bbox.xmin > {SF_XMIN}
    LIMIT 1
""").df()

print(f"Total columns in raw Overture Places schema: {len(schema_df)}")
schema_df

Total columns in raw Overture Places schema: 19


,column_name,column_type,null,key,default,extra
0,id,VARCHAR,YES,None,None,None
1,geometry,GEOMETRY,YES,None,None,None
2,bbox,"STRUCT(xmin FLOAT, xmax FLOAT, ymin FLOAT, yma...",YES,None,None,None
3,version,INTEGER,YES,None,None,None
4,sources,"STRUCT(property VARCHAR, dataset VARCHAR, lice...",YES,None,None,None
5,names,"STRUCT(""primary"" VARCHAR, common MAP(VARCHAR, ...",YES,None,None,None
6,categories,"STRUCT(""primary"" VARCHAR, alternate VARCHAR[])",YES,None,None,None
7,basic_category,VARCHAR,YES,None,None,None
8,taxonomy,"STRUCT(""primary"" VARCHAR, hierarchy VARCHAR[],...",YES,None,None,None
9,confidence,DOUBLE,YES,None,None,None


Notice the nested types: `names` is a STRUCT, `categories` is a STRUCT, `addresses` is a LIST of STRUCTs. The key columns for Scout are:
- `names.primary` — the display name
- `basic_category` — the flattened primary category (Overture added this convenience column in recent releases)
- `geometry` — the WKB point geometry
- `bbox` — the bounding box struct (used for fast S3 filtering before geometry evaluation)
- `confidence` — a 0–1 score indicating data quality

Our ETL will extract just these 6 fields and discard the rest.

## 2. How Many Places Are in SF?

Before we build the full dataset, let's get a row count. The bbox filter on the struct columns (`bbox.xmin`, etc.) is pushed down to S3 as a predicate on Parquet row-group statistics — DuckDB only fetches row groups that *could* overlap the SF bbox.

In [4]:
count_df = con.execute(f"""
    SELECT COUNT(*) AS total_places
    FROM read_parquet('{PLACES_URL}')
    WHERE
        bbox.xmin > {SF_XMIN} AND bbox.xmax < {SF_XMAX} AND
        bbox.ymin > {SF_YMIN} AND bbox.ymax < {SF_YMAX}
""").df()

total = count_df['total_places'][0]
print(f"Total raw places in SF bounding box: {total:,}")
print(f"This is the universe before quality filtering.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total raw places in SF bounding box: 52,179
This is the universe before quality filtering.


## 3. Sample Records

Let's look at real records. This shows us what `names.primary` and `basic_category` look like in practice — and confirms the struct field access syntax works.

In [5]:
sample_df = con.execute(f"""
    SELECT
        id,
        names.primary                   AS name,
        basic_category                  AS category,
        ROUND(confidence, 3)            AS confidence,
        ROUND(bbox.xmin, 5)             AS lon,
        ROUND(bbox.ymin, 5)             AS lat
    FROM read_parquet('{PLACES_URL}')
    WHERE
        bbox.xmin > {SF_XMIN} AND bbox.xmax < {SF_XMAX} AND
        bbox.ymin > {SF_YMIN} AND bbox.ymax < {SF_YMAX} AND
        names.primary IS NOT NULL AND
        basic_category IS NOT NULL
    LIMIT 20
""").df()

sample_df

,id,name,category,confidence,lon,lat
0,7aa264d3-4d60-4dca-a8c9-5bebb7415c0d,Affiliate Supplements,specialty_store,0.570,-122.519547,37.822800
1,d2f7d0ef-8b19-4b4c-ad3e-26af026b8a47,Marin Headlands Institute,event_space,0.770,-122.499062,37.837460
2,4bd12903-8333-46e2-9571-e7920936179b,Presidio Riding Club,social_or_community_service,0.639,-122.499802,37.837730
3,a8cfcb7a-3acd-4b7b-9eef-4f42f0eee981,Little Giggles Day Care,child_care_or_day_care,0.323,-122.500183,37.838249
4,3affdcb2-bca7-414c-b90f-f7938e8e48a6,NatureBridge at Golden Gate Recreational Area,historic_site,0.639,-122.485779,37.839081
5,f09b55a6-19cc-4aa7-8c5a-e6d50da4892e,Bunker Road Tunnel,hotel,0.323,-122.485527,37.838829
6,0fd7766f-74f3-4699-b9ec-e90f19a5866c,Marin Headlands Federal Park,park,0.639,-122.485352,37.838799
7,b40fc29b-fc22-40ac-b70e-3fd78d3a311b,Muir Woods National Park,park,0.639,-122.486122,37.838009
8,d1337dd5-eb61-4293-a562-c4533cc8e53f,Cronkite Beach,beach,0.974,-122.483704,37.838020
9,9e9f594c-ac56-4218-84de-392608596efb,Institute at the Golden Gate,conservation_organization,0.947,-122.482208,37.838051


## 4. Category Distribution

The category values are critical — they become the **closed vocabulary** in the LLM system prompt. If the prompt says "valid categories are: coffee_shop, park, ..." the LLM can only generate SQL that uses real values. This prevents hallucinated category names that would produce empty results.

Let's see the full distribution for SF.

In [6]:
categories_df = con.execute(f"""
    SELECT
        basic_category  AS category,
        COUNT(*)        AS count
    FROM read_parquet('{PLACES_URL}')
    WHERE
        bbox.xmin > {SF_XMIN} AND bbox.xmax < {SF_XMAX} AND
        bbox.ymin > {SF_YMIN} AND bbox.ymax < {SF_YMAX} AND
        basic_category IS NOT NULL
    GROUP BY basic_category
    ORDER BY count DESC
""").df()

print(f"Total distinct categories in SF: {len(categories_df)}")
print()
categories_df.head(30)

Total distinct categories in SF: 364



,category,count
0,restaurant,4270
1,specialty_store,1343
2,social_or_community_service,1288
3,financial_service,1126
4,doctors_office,1103
5,healthcare_location,1074
6,real_estate_agency,880
7,bar,873
8,service_location,791
9,clothing_store,789


In [7]:
# The full sorted list — this is what goes into the STAC catalog
# and ultimately into the LLM prompt
all_categories = sorted(categories_df['category'].tolist())

print(f"All {len(all_categories)} categories (alphabetical):")
print("\n".join(f"  - {c}" for c in all_categories))

All 364 categories (alphabetical):
  - accommodation
  - accountant_or_bookkeeper
  - adoption_service
  - agricultural_area
  - agricultural_service
  - air_transport_facility_service
  - airport
  - airport_terminal
  - allergy_and_immunology
  - alternative_medicine
  - amateur_sport_league
  - amateur_sport_team
  - ambulance_ems_service
  - amusement_park
  - animal_boarding
  - animal_breeding
  - animal_hospital
  - animal_rescue
  - animal_service
  - animal_shelter
  - animal_training
  - antique_shop
  - apartment_building
  - applicance_repair_service
  - aquarium
  - arcade
  - architectural_design_service
  - art_craft_hobby_store
  - art_gallery
  - art_museum
  - astrological_advising
  - atm
  - attorney_or_law_firm
  - auto_body_shop
  - auto_detailing_service
  - auto_glass_service
  - auto_rental_service
  - auto_repair_service
  - b2b_service
  - b2b_supplier_distributor
  - bail_bonds_service
  - bakery
  - bank
  - bar
  - bar_and_grill
  - barber_shop
  - beach
 

## 5. Data Quality Check

Not every record is useful for Scout. We want to understand:
- How many have a name (essential for display)?
- How many have a category (essential for filtering)?
- What does the confidence distribution look like?

These numbers inform our ETL filter threshold.

In [8]:
quality_df = con.execute(f"""
    SELECT
        COUNT(*)                                        AS total,
        COUNT(names.primary)                            AS has_name,
        COUNT(basic_category)                           AS has_category,
        COUNT(*) FILTER (WHERE names.primary IS NOT NULL
                         AND   basic_category IS NOT NULL)
                                                        AS has_both,
        COUNT(*) FILTER (WHERE names.primary IS NOT NULL
                         AND   basic_category IS NOT NULL
                         AND   confidence >= 0.7)       AS scout_eligible,
        ROUND(AVG(confidence), 3)                       AS avg_confidence
    FROM read_parquet('{PLACES_URL}')
    WHERE
        bbox.xmin > {SF_XMIN} AND bbox.xmax < {SF_XMAX} AND
        bbox.ymin > {SF_YMIN} AND bbox.ymax < {SF_YMAX}
""").df()

quality_df

,total,has_name,has_category,has_both,scout_eligible,avg_confidence
0,52179,52179,52179,52179,42492,0.805


In [9]:
# Confidence score histogram — where's the natural cutoff?
conf_df = con.execute(f"""
    SELECT
        ROUND(confidence, 1) AS confidence_bucket,
        COUNT(*)             AS count
    FROM read_parquet('{PLACES_URL}')
    WHERE
        bbox.xmin > {SF_XMIN} AND bbox.xmax < {SF_XMAX} AND
        bbox.ymin > {SF_YMIN} AND bbox.ymax < {SF_YMAX} AND
        confidence IS NOT NULL
    GROUP BY confidence_bucket
    ORDER BY confidence_bucket
""").df()

conf_df

,confidence_bucket,count
0,0.0,3
1,0.1,32
2,0.2,26
3,0.3,5407
4,0.4,708
5,0.5,29
6,0.6,3472
7,0.7,32
8,0.8,16554
9,0.9,2670


## 6. Schema Decision: What to Keep

Based on this exploration, here's what `02_build_sf_places.py` will do:

| Decision | Rationale |
| --- | --- |
| Keep `confidence >= 0.7` | Removes low-quality records while keeping the majority |
| Require `name IS NOT NULL` | Unnamed places aren't useful for display |
| Require `category IS NOT NULL` | Uncategorized places can't be queried by type |
| Flatten `names.primary` → `name` | Simpler schema = fewer LLM hallucination opportunities |
| Use `basic_category` → `category` | Already flattened by Overture; no struct access needed |
| Extract `lat`, `lon` from centroid | Proximity queries need scalar coordinates |
| Store `geometry` as WKB | DuckDB-Wasm's spatial extension reads this format natively |
| Sort by Hilbert curve | Spatial locality → row group skipping in DuckDB-Wasm |
| Compress with ZSTD | Best ratio for our read-heavy, low-write workload |

In [10]:
# Final sanity check: preview the exact query we'll run in the ETL script
preview_df = con.execute(f"""
    SELECT
        id,
        names.primary                        AS name,
        basic_category                       AS category,
        ROUND(ST_Y(ST_Centroid(geometry)), 6) AS lat,
        ROUND(ST_X(ST_Centroid(geometry)), 6) AS lon,
        ST_AsWKB(geometry)                   AS geometry
    FROM read_parquet('{PLACES_URL}')
    WHERE
        bbox.xmin > {SF_XMIN} AND bbox.xmax < {SF_XMAX} AND
        bbox.ymin > {SF_YMIN} AND bbox.ymax < {SF_YMAX} AND
        names.primary IS NOT NULL AND
        basic_category IS NOT NULL AND
        confidence >= 0.7
    LIMIT 10
""").df()

print("Preview of the final ETL output schema:")
preview_df[['id', 'name', 'category', 'lat', 'lon']]  # hide raw geometry bytes

Preview of the final ETL output schema:


,id,name,category,lat,lon
0,d2f7d0ef-8b19-4b4c-ad3e-26af026b8a47,Marin Headlands Institute,event_space,37.837460,-122.499060
1,d1337dd5-eb61-4293-a562-c4533cc8e53f,Cronkite Beach,beach,37.838026,-122.483681
2,9e9f594c-ac56-4218-84de-392608596efb,Institute at the Golden Gate,conservation_organization,37.838055,-122.482194
3,5c957f29-174a-4131-a3f6-f5430a3d503d,Bay Area Discovery Museum,childrens_museum,37.837069,-122.480694
4,cc41dbe3-1b1c-4a72-9614-27bf279d9a80,Murray Circle Restaurant,breakfast_restaurant,37.837575,-122.478799
5,924f504d-c95a-41f2-ab1b-6a9b8cd8e4b2,Sula,restaurant,37.837458,-122.478863
6,563c7cf7-fc77-4f6b-99be-9e6086c70f01,Farley at Cavallo Point,restaurant,37.837517,-122.478847
7,a9b6f40b-a9e4-4c60-a5af-505744599e95,Farley Bar,bar,37.837509,-122.479022
8,50f4f162-9d9d-4cd9-a576-c81fd51d250c,Encore,event_or_party_service,37.837481,-122.479438
9,ebb2d38a-0af0-464d-9df8-d0c5ba886303,Cavallo Point: The Lodge at the Golden Gate,resort,37.837602,-122.479399


In [11]:
con.close()
print("Exploration complete.")
print("Next step: run  uv run python 02_build_sf_places.py")

Exploration complete.
Next step: run  uv run python 02_build_sf_places.py
